**Adding Styles and Unifying the Column Names**

In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
import os
import base64

# --- Step 1: Quebec RSWP Invitations ---
url_quebec = 'https://www.quebec.ca/en/immigration/permanent/skilled-workers/regular-skilled-worker-program/invitation'
response_quebec = requests.get(url_quebec)
soup_quebec = BeautifulSoup(response_quebec.text, 'html.parser')

main_content_quebec = soup_quebec.find('div', class_='main-content') or soup_quebec.body
capture = False
draw_data = []
current_invitation_date = None

for tag in main_content_quebec.find_all(['h2', 'h3', 'p', 'ul']):
    text = tag.get_text(strip=True)
    lower_text = text.lower()
    if "invitation exercises" in lower_text:
        capture = True
        continue
    if capture:
        match_date = re.search(r'Invitations of (.+)', text)
        if match_date:
            current_invitation_date = match_date.group(1).strip()
            continue
        if current_invitation_date:
            score_match = re.search(r'score.*(?:equal to or greater than|of|minimum)?\D*(\d{3,4})\D*', text, re.IGNORECASE)
            score = score_match.group(1) if score_match else ''
            proficiency_match = re.search(r'level\s*(\d+)\s*oral proficiency', text, re.IGNORECASE)
            proficiency = f"Level {proficiency_match.group(1)} oral proficiency" if proficiency_match else ''
            if score or proficiency:
                draw_data.append({
                    'Invitation_Date': current_invitation_date,
                    'Proficiency_in_French': proficiency,
                    'Score': score,
                    'Notes': text
                })

quebec_df = pd.DataFrame(draw_data).fillna('')

# Using correct column names for drop_duplicates
quebec_df = quebec_df.drop_duplicates(subset=["Invitation_Date", "Score"], keep="first")

quebec_content = quebec_df.to_html(index=False, border=1, justify='center')

# --- Step 2: Saskatchewan SINP PDF Data ---
url_sinp_pdf = "https://publications.saskatchewan.ca/api/v1/products/102708/formats/113850/download"
filename_sinp = "SINP_EOI_Selection_Results.pdf"
filepath_sinp = os.path.join('files', filename_sinp)

# Ensure the 'files' directory exists
os.makedirs('files', exist_ok=True)

# Send a GET request to download the PDF
response = requests.get(url_sinp_pdf)

# Checking - if the request was successful
if response.status_code == 200:
    # Save the PDF to the specified file path
    with open(filepath_sinp, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded SINP EOI draw results to {filepath_sinp}")
else:
    print(f"Failed to download SINP PDF. Status code: {response.status_code}")

# --- Step 3: BC PNP Invitations ---
url_bc_pnp = 'https://www.welcomebc.ca/immigrate-to-b-c/invitations-to-apply'
headers = {'User-Agent': 'Mozilla/5.0'}
response_bc_pnp = requests.get(url_bc_pnp, headers=headers)
soup_bc_pnp = BeautifulSoup(response_bc_pnp.content, 'html.parser')

tables_bc_pnp = soup_bc_pnp.find_all('table')

def parse_bc_table(table, is_general=True):
    rows = []
    current_date = ''
    current_draw_type = 'General' if is_general else 'N/A'

    for row in table.find_all('tr'):
        cells = row.find_all(['td', 'th'])
        texts = [c.get_text(strip=True) for c in cells]

        # Skip rows that don't have enough info
        if len(texts) < 3:
            continue

        # General draw table (first one)
        if is_general:
            if len(texts) == 5:
                current_date, current_draw_type, stream, score, invites = texts
            elif len(texts) == 4:
                current_draw_type, stream, score, invites = texts
            elif len(texts) == 3:
                stream, score, invites = texts
            else:
                continue
        # Targeted draw table (second one)
        else:
            if len(texts) == 4:
                current_date, stream, score, invites = texts
            elif len(texts) == 3:
                stream, score, invites = texts
            else:
                continue

        rows.append([current_date.strip(),
            current_draw_type.strip(),
            stream.strip(),
            score.strip(),
            invites.strip()
        ])

    return rows

# Parse both tables
rows1 = parse_bc_table(tables_bc_pnp[0], is_general=True)
rows2 = parse_bc_table(tables_bc_pnp[1], is_general=False)

# Combine both tables
bc_df = pd.DataFrame(rows1 + rows2, columns=["Invitation_Date", "Draw_Type", "Stream", "Cut-off_Score", "Invitations_Issued"])
bc_df = bc_df.drop_duplicates()
bc_df = bc_df[bc_df["Stream"] != "Stream"]
bc_content = bc_df.to_html(index=False, border=1, justify='center')

# First table
table1 = tables_bc_pnp[0]
rows1 = []
current_date = ''
current_draw_type = ''
for row in table1.find_all('tr'):
    cells = row.find_all('td')
    if not cells:
        continue
    texts = [c.get_text(strip=True) for c in cells]

    # Full row
    if len(texts) == 5:
        current_date, current_draw_type, stream, score, invites = texts
    elif len(texts) == 4:
        current_draw_type, stream, score, invites = texts
    elif len(texts) == 3:
        stream, score, invites = texts
    else:
        continue  # skip unexpected format

    rows1.append([current_date, current_draw_type, stream, score, invites])

df1 = pd.DataFrame(rows1, columns=["Invitation_Date", "Draw_Type", "Stream", "Cut-off_Score", "Invitations_Issued"])

# Second table
table2 = tables_bc_pnp[1]
rows2 = []
current_date = ''
for row in table2.find_all('tr'):
    cells = row.find_all('td')
    if not cells:
        continue  # skip header or empty rows
    texts = [c.get_text(strip=True) for c in cells]
    if len(texts) == 4:
        current_date, stream, score, invites = texts
    elif len(texts) == 3:
        stream, score, invites = texts
    rows2.append([current_date, "N/A", stream, score, invites])

df2 = pd.DataFrame(rows2, columns=["Invitation_Date", "Draw_Type", "Stream", "Cut-off_Score", "Invitations_Issued"])

# Merging data
bc_content = pd.concat([df1, df2], ignore_index=True).drop_duplicates()

# --- Step 4: Express Entry Rounds ---
url_express_entry = "https://www.canada.ca/content/dam/ircc/documents/json/ee_rounds_123_en.json"
data_express_entry = requests.get(url_express_entry).json()

records = []
for round in data_express_entry["rounds"]:
    records.append({
        "Invitation_Date": round["drawDateFull"],
        "Draw_Type": round["drawName"],
        "Invitations_Issued": round["drawSize"],
        "Cut-off_Score": round["drawCRS"]
    })

df_express_entry = pd.DataFrame(records)

# --- Step 5: OINP Table Extraction ---

def extract_oinp_tables(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    tables = soup.find_all('table')

    def get_table_title(table):
        heading_tag = table.find_previous(['h2', 'h3', 'h4'])
        return heading_tag.get_text(strip=True) if heading_tag else "Unnamed Table"

    def standardize_headers(headers):
        """Standardize known OINP table headers."""
        header_map = {
            'Date issued': 'Invitation_Date',
            'Date Issued': 'Invitation_Date',
            'Date profiles created': 'Profile_Created_Date',
            'Number of invitations issued': 'Invitations_Issued',
            'Number of NOIs issued': 'Invitations_Issued',
            'CRS score range': 'Score_Range',
            'Score range': 'Score_Range',
        }
        return [header_map.get(h, h) for h in headers]

    Content_oinp = "<h2>Ontario Immigrant Nominee Program (OINP) Draws</h2>\n"
    for table in tables:
        title = get_table_title(table)
        Content_oinp += f"<h3>{title}</h3>\n<table border='1'>\n"

        headers = table.find_all('th')
        if headers:
            raw_headers = [th.get_text(strip=True) for th in headers]
            unified_headers = standardize_headers(raw_headers)
            Content_oinp += "<tr>" + "".join(f"<th>{h}</th>" for h in unified_headers) + "</tr>\n"

            for row in table.find_all('tr')[1:]:
                cells = row.find_all('td')
                if not cells:
                    continue
                row_cells = [td.get_text(strip=True) for td in cells]
                row_cells += [''] * (len(unified_headers) - len(row_cells))  # Pad if necessary
                Content_oinp += "<tr>" + "".join(f"<td>{cell}</td>" for cell in row_cells) + "</tr>\n"
        Content_oinp += "</table><br>\n"

    return Content_oinp

# Define URL and get the OINP content
oinp_url = 'https://www.ontario.ca/page/ontario-immigrant-nominee-program-oinp-invitations-apply'
Content_oinp = extract_oinp_tables(oinp_url)

# --- Step 6: OINP Express Entry

import requests
from bs4 import BeautifulSoup

def get_table_title(table):
    # Traverse up the DOM to find the nearest previous <h2>, <h3>, or <h4> heading
    heading = table.find_previous(['h2', 'h3', 'h4'])
    return heading.get_text(strip=True) if heading else "Untitled Table"

def standardize_headers(headers):
    # Normalize headers if needed
    return [h.strip() for h in headers]

def fetch_oinp_express_entry_tables():
    url = "https://www.ontario.ca/page/oinp-express-entry-notifications-interest"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    tables = soup.find_all("table")
    print(f"Found {len(tables)} tables.")

    content_oinp_ee = "<h2>Ontario Immigrant Nominee Program (OINP) Draws</h2>\n"

    for table in tables:
        title = get_table_title(table)
        content_oinp_ee += f"<h3>{title}</h3>\n<table border='1'>\n"

        headers = table.find_all('th')
        if headers:
            raw_headers = [th.get_text(strip=True) for th in headers]
            unified_headers = standardize_headers(raw_headers)
            content_oinp_ee += "<tr>" + "".join(f"<th>{h}</th>" for h in unified_headers) + "</tr>\n"

            for row in table.find_all('tr')[1:]:
                cells = row.find_all('td')
                if not cells:
                    continue
                row_cells = [td.get_text(strip=True) for td in cells]
                row_cells += [''] * (len(unified_headers) - len(row_cells))
                content_oinp_ee += "<tr>" + "".join(f"<td>{cell}</td>" for cell in row_cells) + "</tr>\n"

        content_oinp_ee += "</table><br>\n"

    # Optional: write to file
    with open("oinp_express_entry_draws.html", "w", encoding="utf-8") as f:
        f.write(content_oinp_ee)

    print("Saved as 'oinp_express_entry_draws.html'")
    return content_oinp_ee

# Run the function
content_oinp_ee = fetch_oinp_express_entry_tables()

# --- Step 6: Manitoba EOI Draws ---
url = 'https://immigratemanitoba.com/news/'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)

mb_content = ""

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    draws = soup.find_all('div', class_='post-content')

    draw_data = []
    for draw in draws:
        draw_title = draw.find('h2', class_='entry-title')
        if not draw_title:
            continue
        draw_title = draw_title.get_text(strip=True)

        draw_date = draw.find('span', class_='published')
        draw_date = draw_date.get_text(strip=True) if draw_date else 'N/A'
        if not draw.find_all('h3'):
            draw_data.append({
                "EOI Draw": draw_title,
                "Invitation_Date": draw_date,
                "Stream": "N/A",
                "LOA Issued": "N/A",
                "Lowest-Ranked Candidate Invited": "N/A",
            })
            continue
        for stream_header in draw.find_all('h3'):
            stream = stream_header.get_text(strip=True)
            ul = stream_header.find_next('ul')
            p_note = stream_header.find_next('p')
            letters = "N/A"
            score = "N/A"
            if ul:
                for li in ul.find_all('li'):
                    if 'Letters of Advice' in li.get_text():
                        strong = li.find('strong')
                        if strong:
                            letters = strong.get_text(strip=True)
                    elif 'lowest-ranked candidate' in li.get_text():
                        strong = li.find('strong')
                        if strong:
                            score = strong.get_text(strip=True)
            note = "N/A"
            if p_note and "Express Entry profile number" in p_note.get_text():
                note = p_note.get_text(strip=True)

            draw_data.append({
                "EOI Draw": draw_title,
                "Invitation_Date": draw_date,
                "Stream": stream,
                "LOA Issued": letters,
                "Lowest-Ranked Candidate Invited": score,
            })

    # Convert to HTML string
    mb_content = """
    <h2>Manitoba Expression of Interest Draws</h2>
    """
    mb_content += "<table><tr><th>Invitation_Date</th><th>Draw_Type</th><th>Stream</th><th>Invitations_Issued</th><th>Cut-off_Score</th></tr>"
    for row in draw_data:
        mb_content += f"<tr><td>{row['Invitation_Date']}</td><td>EOI</td><td>{row['Stream']}</td><td>{row['LOA Issued']}</td><td>{row['Lowest-Ranked Candidate Invited']}</td></tr>"
    mb_content += "</table>"

# --- Step 7: Alberta Advantage Immigration Program – Processing information ---

def extract_alberta_draws():
    url = "https://www.alberta.ca/aaip-processing-information"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    tables = soup.find_all("table")
    content_alberta = ""
    for idx, table in enumerate(tables):
        headers = [th.get_text(strip=True) for th in table.find_all("th")]
        rows = []
        for tr in table.find_all("tr")[1:]:
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if cells:
                rows.append(cells)

        if headers and rows:
            df = pd.DataFrame(rows, columns=headers)
            content_alberta += f"<h4>Table {idx + 1}</h4>" + df.to_html(index=False, border=1, justify="center")

    return content_alberta

# Make sure this line is included BEFORE the html_layout definition
content_alberta = extract_alberta_draws()


# HTML Layout with content inserted dynamically

html_layout = f"""
<html>
<head>
    <meta charset="UTF-8">
    <title>Combined Immigration Program Draws</title>
    <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@400;500;700&display=swap" rel="stylesheet">
    <style>
        body {{
            font-family: 'Roboto', sans-serif;
            margin: 0;
            padding: 0;
            background-color: #f4f4f9;
            color: #333;
        }}
        h1 {{
            color: #003366;
            text-align: center;
            padding: 20px 0;
            margin-bottom: 0;
            font-size: 2em;
            font-weight: 700;
        }}
        .tabs {{
            display: flex;
            justify-content: center;
            background-color: #003366;
            margin-bottom: 20px;
            border-radius: 10px 10px 0 0;
        }}
        .tabs div {{
            padding: 12px 24px;
            color: white;
            cursor: pointer;
            border-radius: 10px 10px 0 0;
            font-weight: 500;
            transition: background-color 0.3s, transform 0.3s ease;
        }}
        .tabs div:hover, .tabs div.active {{
            background-color: #0066cc;
            box-shadow: 0 6px 12px rgba(0, 0, 0, 0.15);
            transform: translateY(-2px);
        }}
        .tab-content {{
            display: none;
            padding: 20px;
            background-color: #fff;
            border-radius: 8px;
            margin-top: 10px;
            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1);
        }}
        .tab-content.active {{
            display: block;
        }}
        table {{
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            margin-top: 20px;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        th, td {{
            padding: 15px 20px;
            text-align: left;
            font-size: 1.1em;
            border-bottom: 1px solid #eee;
            transition: background-color 0.3s ease, transform 0.3s ease;
        }}
        th {{
            background-color: #003366;
            color: white;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 1px;
            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.05);
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        tr:hover {{
            background-color: #f1f1f1;
            transform: translateY(-1px);
        }}
        tr:last-child td {{
            border-bottom: none;
        }}
        .tab-content h2 {{
            font-size: 1.6em;
            color: #003366;
            margin-bottom: 10px;
        }}
        .tab-content h3 {{
            font-size: 1.3em;
            color: #003366;
            margin-top: 20px;
        }}
        .tab-content p {{
            font-size: 1.1em;
            margin-top: 10px;
        }}
        .download-link {{
            display: inline-block;
            margin-top: 20px;
            padding: 12px 25px;
            background-color: #0066cc;
            color: white;
            text-decoration: none;
            border-radius: 25px;
            font-weight: 500;
            font-size: 1.1em;
            transition: background-color 0.3s ease;
        }}
        .download-link:hover {{
            background-color: #005bb5;
        }}
    </style>
    <script>
        window.onload = function() {{
            // Automatically activate the first tab when the page loads
            document.getElementsByClassName('tab')[0].classList.add('active');
            document.getElementsByClassName('tab-content')[0].classList.add('active');
        }}

        function openTab(event, tabName) {{
            const contents = document.getElementsByClassName('tab-content');
            for (let content of contents) content.classList.remove('active');
            const tabs = document.getElementsByClassName('tab');
            for (let tab of tabs) tab.classList.remove('active');
            document.getElementById(tabName).classList.add('active');
            event.currentTarget.classList.add('active');
        }}
    </script>
</head>
<body>

    <h1>Immigration Program Draw Results</h1>

    <div class="tabs">
        <div class="tab" onclick="openTab(event, 'bc_pnp')">British Columbia</div>
        <div class="tab" onclick="openTab(event, 'aaip')">Alberta</div>
        <div class="tab" onclick="openTab(event, 'express_entry')">Express Entry</div>
        <div class="tab" onclick="openTab(event, 'oinp')">Ontario</div>
        <div class="tab" onclick="openTab(event, 'oinp_ee')">Ontario Express Entry</div>
        <div class="tab" onclick="openTab(event, 'quebec')">Quebec</div>
        <div class="tab" onclick="openTab(event, 'manitoba')">Manitoba</div>
        <div class="tab" onclick="openTab(event, 'sinp')">Saskatchewan</div>
    </div>

    <div id="quebec" class="tab-content">
        <h2>Quebec Regular Skilled Worker Program Invitations</h2>
        {quebec_content}
    </div>

    <div id="bc_pnp" class="tab-content">
        <h2>British Columbia Draws</h2>
        <h3>General and Targeted Draws</h3>
        {df1.to_html(index=False, border=1)}
        <h3>Sector-Specific Draws</h3>
        {df2.to_html(index=False, border=1)}
    </div>

    <div id="sinp" class="tab-content">
        <h2>Saskatchewan Draw Results</h2>
        <p>Download the latest SINP EOI Selection Results:</p>
        <a href="files/SINP_EOI_Selection_Results.pdf" class="download-link" download>Download SINP PDF</a>
    </div>

    <div id="express_entry" class="tab-content">
        <h2>Express Entry Rounds</h2>
        {df_express_entry.to_html(index=False, border=1)}
    </div>

    <div id="oinp" class="tab-content">
        <h2>Ontario Immigrant Nominee Program (OINP)</h2>
        {Content_oinp}
    </div>

    <div id="manitoba" class="tab-content">
        {mb_content}
    </div>

    <div id="aaip" class="tab-content">
        <h2>Alberta Advantage Immigration Program – Processing information</h2>
        {content_alberta}
    </div>

    <div id="oinp_ee" class="tab-content">
        <h2>OINP Express Entry</h2>
        {content_oinp_ee}
    </div>


</body>
</html>
"""

# Saving the final HTML to a file
output_filename = "combined_immigration_program_results.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(html_layout)

#Downloading The File
from google.colab import files
files.download(output_filename)
print(f"HTML saved as {output_filename}")

Downloaded SINP EOI draw results to files/SINP_EOI_Selection_Results.pdf
Found 21 tables.
Saved as 'oinp_express_entry_draws.html'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

HTML saved as combined_immigration_program_results.html
